# 03. QUBO 생성 및 정확성 검증

## penalty coefficient 계산

SS와 MS의 목적함수 계수는 모두 비음수이다 (`f_j >= 0`, `c_ij d_i >= 0`). 따라서 목적함수에서 얻을 수 있는 최대 이득의 보수적 상한은

$$U_{obj} = \sum_j f_j + \sum_i \sum_j c_{ij} d_i$$

이다. 정수 계수를 갖는 등식 제약 `g_k(z) = 0`이 위반되면 `g_k(z)^2 >= 1`이므로 penalty 증가분은 최소 `lambda`이다. 따라서

$$\lambda = \text{margin} \times U_{obj}, \qquad \text{margin} > 1$$

로 두면 어떤 제약 위반도 목적함수 개선으로 상쇄될 수 없다. margin은 설정 파일에 고정하며 grid search 하지 않는다.

MS QUBO에서 `q_ij`의 binary expansion 계수 합이 정확히 `d_i`이므로 운송비 계수 총합이 SS와 같아진다. 즉 SS와 MS가 **동일한 기준식**으로 lambda를 계산한다.

In [ ]:
# 프로젝트 루트를 import 경로에 추가한다.
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.config import load_config, resolve_path

config = load_config(PROJECT_ROOT / "config" / "experiment_config.yaml")
DATA_DIR = resolve_path(config, "data_dir")
RAW_DIR = resolve_path(config, "raw_dir")
PROCESSED_DIR = resolve_path(config, "processed_dir")
FIGURE_DIR = resolve_path(config, "figure_dir")
SOLUTION_DIR = RAW_DIR / "solutions"
SOLUTION_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)
print("설정 로드 완료:", len(config["instances"]), "개 instance")


In [ ]:
from src.data_generator import CFLPInstance
from src.persistence import load_solution, save_table
from src.qubo_builder import build_qubo, qubo_statistics

margin = float(config["penalty"]["margin"])
precision = int(config["encoding"]["precision"])
instances = [
    CFLPInstance.load(DATA_DIR / f"{spec['name']}.json")
    for spec in config["instances"]
]

models = {}
rows = []
for instance in instances:
    for formulation in ("SS", "MS"):
        model = build_qubo(instance, formulation, margin, precision)
        models[(instance.name, formulation)] = model
        row = {
            "instance": instance.name,
            "size": instance.num_customers,
            "formulation": formulation,
        }
        row.update(qubo_statistics(model))
        rows.append(row)

qubo_stats = pd.DataFrame(rows)
save_table(qubo_stats, RAW_DIR, "qubo_stats.csv")
qubo_stats

## 변수 수 구성

- `decision_variables`: 원래 formulation에 이미 binary인 변수
- `encoding_variables`: 정수 변수 `q_ij`의 binary expansion 변수
- `slack_variables`: capacity 부등식을 등식으로 바꾸기 위한 slack 변수

In [ ]:
pivot = qubo_stats.pivot_table(
    index=["instance", "size"],
    columns="formulation",
    values=["qubo_variables", "qubo_terms"],
).sort_index(level="size")
pivot["MS/SS variable ratio"] = (
    pivot[("qubo_variables", "MS")] / pivot[("qubo_variables", "SS")]
).round(2)
pivot

## 검증 1 — binary expansion의 표현 완전성

마지막 계수를 `U - (2^(K-1) - 1)`로 잘라낸 encoding이 `[0, U]`의 **모든 정수**를 정확히 표현하는지 전수 확인한다. 이 검증이 없으면 encoding이 표현할 수 없는 값 때문에 ground state가 구조적으로 infeasible해질 수 있다.

In [ ]:
from src import validation

bounds = sorted(
    {int(value) for instance in instances for value in instance.demands}
    | {int(value) for instance in instances for value in instance.capacities}
    | set(range(0, 17))
)
check = validation.validate_encoding_completeness(bounds)
print(check)
assert check.passed

## 검증 2 — energy 항등식 (random validation)

무작위 binary 할당 `z`에 대해 다음이 성립해야 한다.

$$\text{QUBO energy}(z) = f_{orig}(\text{decode}(z)) + \lambda \sum_k g_k(z)^2$$

상수 offset 누락, 계수 합산 오류, 부호 오류가 있으면 즉시 드러난다.

## 검증 3 — reference solution encoding

Gurobi 최적해를 QUBO 변수로 encoding했을 때 모든 제약 잔차가 0이고 QUBO energy가 MILP 목적값과 일치해야 한다.

In [ ]:
tolerance = float(config["validation"]["tolerance"])
num_samples = int(config["validation"]["random_samples"])
seed = int(config["validation"]["random_seed"])

validation_rows = []
for instance in instances:
    for formulation in ("SS", "MS"):
        model = models[(instance.name, formulation)]
        reference_tag = "gurobi_SS" if formulation == "SS" else "gurobi_MS_INT"
        reference_solution = load_solution(SOLUTION_DIR, instance.name, reference_tag)
        reference_row = pd.read_csv(RAW_DIR / "gurobi_results.csv")
        wanted = "SS" if formulation == "SS" else "MS-int-q"
        reference_objective = float(
            reference_row[
                (reference_row["instance"] == instance.name)
                & (reference_row["gurobi_model"] == wanted)
            ]["objective"].iloc[0]
        )
        for result in (
            validation.validate_energy_identity(
                model, instance, num_samples, seed, tolerance
            ),
            validation.validate_reference_solution(
                model, instance, reference_solution, reference_objective, tolerance
            ),
        ):
            print(result)
            validation_rows.append(
                {
                    "instance": instance.name,
                    "formulation": formulation,
                    "check": result.name.split("[")[0],
                    "passed": result.passed,
                    "message": result.message,
                }
            )

## 검증 4 — exhaustive validation

실제 실험 instance는 QUBO 변수가 44개 이상이라 전수 열거가 불가능하다 (`2^44` 이상). 따라서 다음 두 가지를 수행한다.

1. 실제 instance: 전수 열거를 시도하되 한계를 넘으면 사유와 함께 SKIP 기록
2. 별도의 2x2 toy instance: 수요/용량을 매우 작게 고정하여 **전수 열거로** QUBO 최소값이 MILP 최적값과 일치하는지 확인

toy instance는 실제 실험 결과에 포함되지 않으며 오직 변환 정확성 검증에만 사용한다.

In [ ]:
from src import gurobi_solver
from src.data_generator import build_toy_instance

toy = build_toy_instance(config["validation"]["toy"])
toy_references = {
    "SS": gurobi_solver.solve_ss(toy, config["gurobi"]),
    "MS": gurobi_solver.solve_ms_integer(toy, config["gurobi"]),
}
for formulation, reference in toy_references.items():
    toy_model = build_qubo(toy, formulation, margin, precision)
    result = validation.validate_exhaustive(
        toy_model, toy, reference.objective, tolerance
    )
    print(f"toy {formulation}: QUBO 변수 {toy_model.num_variables}개")
    print("   ", result)
    validation_rows.append(
        {
            "instance": "toy2x2",
            "formulation": formulation,
            "check": "exhaustive",
            "passed": result.passed,
            "message": result.message,
        }
    )

In [ ]:
for instance in instances:
    for formulation in ("SS", "MS"):
        model = models[(instance.name, formulation)]
        result = validation.validate_exhaustive(model, instance, 1.0, tolerance)
        validation_rows.append(
            {
                "instance": instance.name,
                "formulation": formulation,
                "check": "exhaustive",
                "passed": result.passed,
                "message": result.message,
            }
        )
        print(result)

In [ ]:
validation_log = pd.DataFrame(validation_rows)
save_table(validation_log, RAW_DIR, "validation_log.csv")
print(
    f"검증 {len(validation_log)}건 중 통과 "
    f"{int(validation_log['passed'].sum())}건"
)
assert bool(validation_log["passed"].all()), "검증 실패 항목이 있습니다."
validation_log

## QUBO 계수 통계

penalty가 목적함수 규모를 압도하도록 설정되므로 계수의 dynamic range가 매우 커진다. 이는 QA 하드웨어의 아날로그 정밀도와 직결되는 문제이며 RQ5의 핵심 관찰 대상이다.

In [ ]:
qubo_stats[
    [
        "instance",
        "formulation",
        "penalty_lambda",
        "qubo_min",
        "qubo_max",
        "qubo_range",
    ]
].assign(
    dynamic_range_orders=lambda frame: np.log10(
        frame["qubo_range"].abs().clip(lower=1e-12)
    ).round(2)
)